# Avance 5: Modelo Final — MLP Supervisado

**Proyecto integrador** — conjunto de datos **InHARD**.
## AI Co-Pilot for the Production Floor

**Equipo #56:** Landy Haydee Schlebach Osorio, Carlos Pano Hernández, Carlos Fernando Del Castillo Rey

**Sponsor:** Dr. Jacobo Eluani

**Asesor Académico**: Dr. Gerardo Camacho

---

En los experimentos anteriores detectamos inactividad usando similitud coseno con prototipos — un enfoque zero-shot que no requiere entrenamiento. Este experimento incorpora un **clasificador MLP supervisado** entrenado sobre embeddings fpc64 multi-vista, replicando la arquitectura que el equipo validó con 81% de accuracy sobre 12 clases.

La diferencia clave respecto al experimento anterior es el clasificador:
- Experimentos anteriores: similitud coseno vs. prototipos (sin entrenamiento)
- Este experimento: MLP entrenado sobre embeddings de InHARD (supervisado)

Las 12 clases incluyen "No action", por lo que el MLP clasifica directamente inactividad y las 11 acciones de manufactura restantes — sin necesidad de un detector binario separado.


# 1. Importar Librerías

In [ ]:
import cv2
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

import torch
from transformers import AutoModel, AutoVideoProcessor
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Device: {DEVICE}")


# 2. Configuración
Definimos los parámetros clave para la ejecución.

### Modelo
Mismo modelo fpc64 y misma fusión de tres vistas que en el experimento anterior. La diferencia está en el clasificador que se entrena sobre esos embeddings.

### Dataset de entrenamiento
Para cada una de las 12 clases de InHARD tomamos los primeros `N_TRAIN_PER_CLASS` clips disponibles de cualquier participante. Esto es el conjunto sobre el que se entrena el MLP. Los clips de evaluación de P05 se mantienen separados — el MLP nunca los ve durante el entrenamiento.

### MLP
Red neuronal de dos capas ocultas (256 y 128 neuronas), función de activación ReLU, regularización L2 con alpha=0.001. La misma arquitectura que produjo 81% en los experimentos del equipo.


In [ ]:
from pathlib import Path

NB10_EMBEDDINGS = Path("../outputs/10_action_id_v2/embeddings_v2.npy")
NB10_LABELS     = Path("../outputs/10_action_id_v2/labels_v2.npy")
INHARD_ROOT     = Path(
    "/Users/landyschlebach/Dataset In Hard/01-InHARD.7z/Segmented/RGBSegmented"
)

PARTICIPANT        = "P05"
IDLE_CLASS         = "No action"
N_FRAMES           = 8
N_VIEWS            = 3
N_TRAIN_PER_CLASS  = 8    # clips por clase para entrenamiento
N_EVAL_IDLE        = 8
N_EVAL_ACTIVE      = 26

# Limpieza de clases (consistente con experimentos del equipo)
MERGE_MAP = {
    "Take subsystem":      "Assemble system",
    "Put down subsystem":  "Assemble system",
}

# Hiperparámetros del MLP
MLP_HIDDEN    = (256, 128)
MLP_ALPHA     = 0.001
MLP_MAX_ITER  = 500

OUTPUTS    = Path("outputs/24_mlp_supervisado")
OUTPUTS.mkdir(parents=True, exist_ok=True)
CACHE_TRAIN = OUTPUTS / "embeddings_train.npz"
CACHE_EVAL  = OUTPUTS / "embeddings_eval.npz"

print(f"Ruta InHARD:                         {INHARD_ROOT}")
print(f"Participante de evaluacion:          {PARTICIPANT}")
print(f"Modelo:                              fpc64 | {N_VIEWS} vistas | fusion promedio")
print(f"Frames de muestreo:                  {N_FRAMES}")
print(f"Clips de entrenamiento por clase:    {N_TRAIN_PER_CLASS}")
print(f"MLP arquitectura:                    {MLP_HIDDEN}")
print(f"Clips reservados para evaluacion:    {N_EVAL_IDLE} idle + {N_EVAL_ACTIVE} active")


# 3. Funciones Core

Las mismas funciones de extracción multi-vista del experimento anterior. `crop_view()` extrae uno de los tres tercios del mosaico InHARD y `load_clip_frames_multiview()` carga los frames de las tres vistas simultáneamente.


In [ ]:
def crop_view(frame_bgr, view_idx):
    H, W = frame_bgr.shape[:2]
    w    = W // 3
    crop = frame_bgr[:, view_idx*w:(view_idx+1)*w, :]
    return cv2.resize(crop, (224, 224))

def load_clip_frames_multiview(clip_path, n=N_FRAMES):
    cap   = cv2.VideoCapture(str(clip_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release()
        return [[] for _ in range(N_VIEWS)]
    idx   = np.linspace(0, total-1, n, dtype=int)
    views = [[] for _ in range(N_VIEWS)]
    for i in idx:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, f = cap.read()
        if ret:
            for v in range(N_VIEWS):
                views[v].append(crop_view(f, v))
    cap.release()
    return views

print("Funciones core definidas.")


# 4. V-JEPA2

Modelo fpc64 congelado, mismo que el experimento anterior. El backbone nunca se modifica — solo lo usamos para extraer embeddings de 1024 dimensiones que el MLP aprende a clasificar.


In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

MODEL_ID = "facebook/vjepa2-vitl-fpc64-256"
print(f"Cargando {MODEL_ID}...")

vj_proc  = AutoVideoProcessor.from_pretrained(MODEL_ID)
vj_model = AutoModel.from_pretrained(MODEL_ID).to(DEVICE).eval()

print(f"V-JEPA2 listo en {DEVICE}.")

def embed_clip_single(frames_bgr):
    if not frames_bgr:
        return None
    try:
        rgb = [cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in frames_bgr]
        inp = vj_proc(list(np.stack(rgb, 0)), return_tensors="pt")
        inp = {k: v.to(DEVICE) for k, v in inp.items()}
        with torch.inference_mode():
            out = vj_model(**inp)
        if hasattr(out, "last_hidden_state"):
            raw = out.last_hidden_state.mean(dim=1).squeeze(0)
        else:
            raw = out.pooler_output.squeeze(0)
        emb  = raw.cpu().numpy().astype(np.float32)
        norm = np.linalg.norm(emb)
        return emb / (norm + 1e-8) if norm > 1e-8 else emb
    except Exception:
        return None

def embed_clip_multiview(frames_per_view):
    embs = []
    for view_frames in frames_per_view:
        emb = embed_clip_single(view_frames)
        if emb is not None:
            embs.append(emb)
    if not embs:
        return None
    fused = np.mean(embs, axis=0)
    norm  = np.linalg.norm(fused)
    return fused / (norm + 1e-8) if norm > 1e-8 else fused


# 5. Dataset de Entrenamiento

Para entrenar el MLP necesitamos embeddings fpc64 multi-vista de múltiples participantes y las 12 clases limpias. Tomamos los primeros `N_TRAIN_PER_CLASS` clips de cada clase — sin filtrar por participante, para que el modelo aprenda variabilidad entre personas.

Estos clips son completamente distintos a los 34 clips de evaluación de P05, garantizando que la evaluación sea honesta.


In [ ]:
# Clases disponibles tras la limpieza
all_action_dirs = [d for d in INHARD_ROOT.iterdir()
                   if d.is_dir() and d.name not in ("RGB",)]

# Aplicar merge map y construir lista de clases limpias
raw_class_names = sorted(d.name for d in all_action_dirs)
clean_names     = sorted(set(MERGE_MAP.get(c, c) for c in raw_class_names))
print(f"Clases originales: {len(raw_class_names)}")
print(f"Clases tras limpieza: {len(clean_names)}")
print(f"Clases: {clean_names}")

def get_clips_any_participant(action_folder, max_n):
    """Toma los primeros max_n clips de cualquier participante."""
    clips = sorted(action_folder.glob("*.mp4"))
    if not clips:
        clips = sorted(action_folder.rglob("*.mp4"))
    return clips[:max_n]


In [ ]:
if CACHE_TRAIN.exists():
    print("Cargando dataset de entrenamiento desde cache...")
    data       = np.load(CACHE_TRAIN, allow_pickle=True)
    X_train    = data["X_train"]
    y_train    = data["y_train"]
    print(f"  X_train: {X_train.shape}  |  clases: {sorted(set(y_train.tolist()))}")
else:
    X_list, y_list = [], []
    for raw_cls in tqdm(raw_class_names, desc="Extrayendo por clase"):
        clean_cls = MERGE_MAP.get(raw_cls, raw_cls)
        folder    = INHARD_ROOT / raw_cls
        clips     = get_clips_any_participant(folder, N_TRAIN_PER_CLASS)

        for clip_path in clips:
            views = load_clip_frames_multiview(clip_path)
            emb   = embed_clip_multiview(views)
            if emb is not None:
                X_list.append(emb)
                y_list.append(clean_cls)

    X_train = np.stack(X_list)
    y_train = np.array(y_list)
    np.savez(CACHE_TRAIN, X_train=X_train, y_train=y_train)
    print(f"Cache guardado en {CACHE_TRAIN}")

print(f"\nDistribucion del dataset de entrenamiento:")
for cls in sorted(set(y_train.tolist())):
    n = (y_train == cls).sum()
    print(f"  {cls:<35}: {n} clips")


# 6. Entrenamiento del MLP

Entrenamos el MLP sobre los embeddings fpc64 multi-vista del dataset de entrenamiento. Usamos validación cruzada estratificada de 5 folds para estimar el rendimiento antes de evaluar en el set de P05.

El MLP aprende a separar las 12 clases en el espacio de embeddings — incluyendo "No action". A diferencia del coseno, puede encontrar fronteras no lineales en ese espacio comprimido.


In [ ]:
le  = LabelEncoder()
y_enc = le.fit_transform(y_train)

mlp = MLPClassifier(
    hidden_layer_sizes=MLP_HIDDEN,
    activation="relu",
    alpha=MLP_ALPHA,
    max_iter=MLP_MAX_ITER,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
)

# Validación cruzada
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_acc = cross_val_score(mlp, X_train, y_enc, cv=skf, scoring="accuracy",    n_jobs=-1)
cv_f1  = cross_val_score(mlp, X_train, y_enc, cv=skf, scoring="f1_weighted", n_jobs=-1)

print(f"Validacion cruzada 5-fold:")
print(f"  Accuracy:    {cv_acc.mean():.4f} +/- {cv_acc.std():.4f}")
print(f"  F1-weighted: {cv_f1.mean():.4f}  +/- {cv_f1.std():.4f}")
print(f"  Por fold:    {' '.join([f'{a:.3f}' for a in cv_acc])}")

# Entrenar en todos los datos de entrenamiento
mlp.fit(X_train, y_enc)
print(f"\nMLP entrenado. Iteraciones: {mlp.n_iter_}")

# Accuracy sobre el conjunto de entrenamiento (referencia de capacidad)
train_acc = accuracy_score(y_enc, mlp.predict(X_train))
print(f"Accuracy en entrenamiento: {train_acc:.4f}")
print(f"(Gap train-CV: {train_acc - cv_acc.mean():.4f})")


# 7. Secuencia de Evaluación — P05

Misma separación que en experimentos previos: 8 clips de "No action" de P05 para evaluación y 26 clips de "Assemble system". Estos clips nunca fueron vistos por el MLP durante el entrenamiento.


In [ ]:
idle_dir   = INHARD_ROOT / "No action"
active_dir = INHARD_ROOT / "Assemble system"

def get_clips_participant(folder, pid, max_n):
    clips = sorted(folder.glob(f"{pid}_*.mp4"))
    if not clips:
        clips = sorted(folder.rglob(f"{pid}_*.mp4"))
    return clips[:max_n]

all_idle_p05   = get_clips_participant(idle_dir,   PARTICIPANT, 9999)
all_active_p05 = get_clips_participant(active_dir, PARTICIPANT, 9999)

# Separar proto de eval para mantener consistencia con experimentos anteriores
eval_idle   = all_idle_p05[:N_EVAL_IDLE]
eval_active = all_active_p05[:N_EVAL_ACTIVE]

eval_sequence = ([(c, "Assemble system") for c in eval_active[:20]] +
                 [(c, "No action")       for c in eval_idle]         +
                 [(c, "Assemble system") for c in eval_active[20:26]])

gt_labels = [l for _, l in eval_sequence]
gt_bin    = np.array([1 if l == IDLE_CLASS else 0 for l in gt_labels])

print(f"Clips de evaluacion de P05:")
print(f"  'No action':       {(gt_bin==1).sum()} clips")
print(f"  'Assemble system': {(gt_bin==0).sum()} clips")
print(f"  Total:             {len(eval_sequence)} clips")


# 8. Embeddings de Evaluación

Extraemos los embeddings fpc64 multi-vista de los 34 clips de P05. El proceso es idéntico al del dataset de entrenamiento — misma fusión de tres vistas, mismos 8 frames de muestreo.


In [ ]:
if CACHE_EVAL.exists():
    print("Cargando embeddings de evaluacion desde cache...")
    data   = np.load(CACHE_EVAL, allow_pickle=True)
    X_eval = data["X_eval"]
    print(f"  X_eval: {X_eval.shape}")
else:
    embs = []
    for clip_path, _ in tqdm(eval_sequence, desc="Eval clips"):
        views = load_clip_frames_multiview(clip_path)
        emb   = embed_clip_multiview(views)
        embs.append(emb if emb is not None else np.zeros(1024, dtype=np.float32))
    X_eval = np.stack(embs)
    np.savez(CACHE_EVAL, X_eval=X_eval)
    print(f"Cache guardado en {CACHE_EVAL}")


# 9. Evaluación Final

Aplicamos el MLP entrenado sobre los 34 clips de P05. El modelo predice directamente una de las 12 clases — incluyendo "No action" como clase de inactividad — sin necesidad de un detector binario separado.

Reportamos dos niveles:
1. **Detección de inactividad**: ¿predijo "No action" cuando correspondía?
2. **Clasificación completa**: accuracy sobre las 12 clases en los clips de evaluación


In [ ]:
y_eval_pred_enc = mlp.predict(X_eval)
y_eval_pred     = le.inverse_transform(y_eval_pred_enc)

# ── Nivel 1: detección de inactividad ────────────────────────────────────────
pred_bin = np.array([1 if p == IDLE_CLASS else 0 for p in y_eval_pred])
idle_acc = accuracy_score(gt_bin, pred_bin)

from sklearn.metrics import precision_score, recall_score, f1_score as f1sc
idle_p = precision_score(gt_bin, pred_bin, zero_division=0)
idle_r = recall_score(gt_bin, pred_bin, zero_division=0)
idle_f = f1sc(gt_bin, pred_bin, zero_division=0)

print(f"Deteccion de inactividad (binario):")
print(f"  Precision: {idle_p:.4f}")
print(f"  Recall:    {idle_r:.4f}")
print(f"  F1:        {idle_f:.4f}")
print()

# ── Nivel 2: clasificacion completa ──────────────────────────────────────────
eval_acc = accuracy_score(gt_labels, y_eval_pred)
eval_f1  = f1sc(gt_labels, y_eval_pred, average="weighted", zero_division=0)

print(f"Clasificacion completa (12 clases):")
print(f"  Accuracy:    {eval_acc:.4f}")
print(f"  F1-weighted: {eval_f1:.4f}")
print()
print(classification_report(gt_labels, y_eval_pred, zero_division=0))


In [ ]:
print(f"Prediccion por clip:")
print(f"  {'GT':<25}  {'Prediccion':<25}  Correcto")
print(f"  {'-'*65}")
for (_, gt), pred in zip(eval_sequence, y_eval_pred):
    ok = "OK" if pred == gt else "X "
    print(f"  {gt:<25}  {pred:<25}  {ok}")


# 10. Visualización

El primer panel muestra la matriz de confusión del MLP sobre los clips de evaluación. El segundo compara la progresión histórica de experimentos anteriores con los resultados del MLP supervisado.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: matriz de confusion
present_classes = sorted(set(gt_labels) | set(y_eval_pred))
cm = confusion_matrix(gt_labels, y_eval_pred, labels=present_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=present_classes)
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(f"Matriz de confusion — evaluacion P05\nAccuracy={eval_acc:.3f}")
axes[0].tick_params(axis='x', rotation=45)

# Panel 2: progresion historica
hist_labels  = ["YOLO crop", "ROI central", "ROI izq.", "Proto P05",
                "MV k=3\nfpc16", "Dif+EWMA\nfpc64"]
hist_f1s     = [0.155, 0.354, 0.571, 0.667, 0.750, 0.727]
hist_colors  = ["#ccc"] * 6
mlp_label    = f"MLP\nfpc64"
all_labels   = hist_labels + [mlp_label]
all_f1s      = hist_f1s    + [idle_f]
all_colors   = hist_colors + ["#E84040"]

x = np.arange(len(all_labels))
axes[1].bar(x, all_f1s, color=all_colors, alpha=0.85, width=0.6)
axes[1].axhline(0.9, ls=":", color="green", lw=2, label="Meta 0.90")
axes[1].set_xticks(x)
axes[1].set_xticklabels(all_labels, fontsize=8)
axes[1].set_ylabel("F1 (deteccion de inactividad)")
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Progresion historica — F1 deteccion idle")
axes[1].legend(fontsize=9)
axes[1].grid(axis="y", alpha=0.3)
for bar, v in zip(axes[1].patches, all_f1s):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.02,
                 f"{v:.2f}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUTS / "evaluacion_mlp.png", dpi=130, bbox_inches="tight")
plt.show()


# 11. Resultados

Tabla de progresión completa. Los renglones de experimentos anteriores muestran el F1 de detección de inactividad (binario). Para el MLP reportamos tanto el F1 binario de inactividad como la accuracy de clasificación de 12 clases.


In [ ]:
print(f"Participante evaluacion:   {PARTICIPANT}")
print(f"Modelo:                    fpc64 | {N_VIEWS} vistas | fusion promedio")
print(f"Clasificador:              MLP {MLP_HIDDEN}")
print(f"Clips entrenamiento:       {len(X_train)} ({N_TRAIN_PER_CLASS} por clase)")
print(f"Clips evaluacion:          {len(eval_sequence)}")
print()
print(f"{'Metodo':<40} {'F1 idle':>8}  {'Acc total':>10}")
print("-" * 64)

historico = [
    ("YOLO crop (coseno, fpc16)",             0.155, "-"),
    ("ROI central (coseno, fpc16)",           0.354, "-"),
    ("ROI izq. (coseno, fpc16)",               0.571, "-"),
    ("Proto P05 (coseno, fpc16)",              0.667, "-"),
    ("Mayoria k=3 (coseno, fpc16)",            0.750, "0.880"),
    ("Dif+EWMA (coseno, fpc64, 3 vistas)",    0.727, "0.820"),
]
for lbl, f1, acc in historico:
    print(f"{lbl:<40} {f1:>8.3f}  {acc:>10}")

print("-" * 64)
acc_str = f"{eval_acc:.3f}"
print(f"{'MLP supervisado (fpc64, 3 vistas)':<40} {idle_f:>8.3f}  {acc_str:>10}")

print()
print(f"Validacion cruzada MLP (5-fold):")
print(f"  Accuracy:    {cv_acc.mean():.4f} +/- {cv_acc.std():.4f}")
print(f"  F1-weighted: {cv_f1.mean():.4f}  +/- {cv_f1.std():.4f}")


# 12. Ajuste de Umbral de Confianza

El MLP devuelve probabilidades por clase. Se puede ajustar un umbral de confianza mínima para la clase "No action" — si la probabilidad no supera el umbral, el sistema reporta "activo" aunque "No action" sea la clase más probable. Modificar `CONF_THRESHOLD` y re-ejecutar.


In [ ]:
CONF_THRESHOLD = 0.0   # 0.0 = sin filtro de confianza

y_proba = mlp.predict_proba(X_eval)
idle_idx_le = list(le.classes_).index(IDLE_CLASS)

pred_thresh = []
for probs in y_proba:
    best_cls = le.classes_[np.argmax(probs)]
    if best_cls == IDLE_CLASS and probs[idle_idx_le] < CONF_THRESHOLD:
        # No hay suficiente confianza — marcar como activo
        active_probs = probs.copy()
        active_probs[idle_idx_le] = 0
        best_cls = le.classes_[np.argmax(active_probs)]
    pred_thresh.append(best_cls)

pred_thresh_bin = np.array([1 if p == IDLE_CLASS else 0 for p in pred_thresh])
acc_thresh = accuracy_score(gt_bin, pred_thresh_bin)
f1_thresh  = f1sc(gt_bin, pred_thresh_bin, zero_division=0)
p_thresh   = precision_score(gt_bin, pred_thresh_bin, zero_division=0)
r_thresh   = recall_score(gt_bin, pred_thresh_bin, zero_division=0)

print(f"Umbral de confianza: {CONF_THRESHOLD}")
print(f"P={p_thresh:.3f}  R={r_thresh:.3f}  F1={f1_thresh:.3f}  Acc={acc_thresh:.3f}")
